# 5. Sweep, part 1: every candidate point (GPU)

Every number so far came from one operating point: keep a local maximum if it
is at least 30% of its patch's own peak, and drop the patch if its peak is
under 0.1. The saved detections kept only what passed, so no other setting
could be examined. This notebook runs the four models once more and keeps
**every local maximum down to 1% of each patch's peak**, which is enough to
replay any setting above that exactly, on CPU, in `06_sweep_curves`.

**Faster than the full run.** That run spent most of its first pass (16
minutes for OWL-D) opening 2,607 files one by one over Drive. Here the split
crosses as one 1.2 GB archive, is checked against its published digest, and is
unpacked on local disk.

**Checked before it is trusted.** For each model, replaying the evaluation
setting must give back the detections `01_full_cah` wrote. If it does not, the
notebook says how many patches differ and stops before anything is built on it.

Standard RAM is enough. Nothing is scored here.


In [1]:
# --- 0. Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# --- 1. Configuration ---
ASSETS  = "/content/drive/MyDrive/OWL_Caribou_Project"
RESULTS = "/content/drive/MyDrive/owl_caribou_overhead/results"
STAGE   = "/content/cah_local"          # fast local disk, rebuilt each runtime

RUN_NAME = "full_cah"
MODELS   = ["owl-d", "caribou-owl-c", "owl-c", "owl-t"]
FLOOR    = 0.01        # keep local maxima down to 1% of each patch's peak
BATCH_SIZE, NUM_WORKERS = 8, 4
PROBE_PATCHES = 32     # candidate density is measured on these before the full pass
MAX_CANDIDATES_PER_PATCH = 20000   # above this, raise FLOOR rather than continue
FORCE = False

from pathlib import Path
if not (Path(ASSETS) / "data" / "test" / "gt.csv").is_file():
    raise RuntimeError(f"{ASSETS}/data/test/gt.csv not found. Put the assets there as the README's "
                       "'Expected assets' section describes, or point ASSETS at where they are.")


In [ ]:
# === CODE SYNC (auto-generated by `python -m owlcaribou.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m owlcaribou.sync   and reopen this notebook.")

In [4]:
# --- 3. Start the session ---
from owlcaribou.session import start_session

session = start_session(assets=ASSETS, results=RESULTS, models=MODELS, require_gpu=True)

session: Colab | device cuda (NVIDIA A100-SXM4-40GB, 39.49 GB)
  torch   2.11.0+cu128 | CUDA 12.8 | Python 3.13.15
  assets  /content/drive/MyDrive/OWL_Caribou_Project
  results /content/drive/MyDrive/owl_caribou_overhead/results
  code    c26c4d3f466e
  split   2607 patches, 12456 points
  weights verified: caribou-owl-c, owl-c, owl-d, owl-t


In [5]:
# --- 4. Put the split on local disk: one archive instead of 2,607 Drive reads ---
import time
from owlcaribou import io as oio
from owlcaribou.data import CahPatches

started = time.perf_counter()
local_split = oio.stage_split(session.paths, STAGE)
patches = CahPatches(local_split, expect_images=2607)
print(f"staged {len(patches)} patches in {time.perf_counter() - started:.0f} s -> {local_split}")

staged 2607 patches in 57 s -> /content/cah_local/extracted


In [6]:
# --- 5. Every model: extract candidates, store them, prove they replay the full run ---
import time
import numpy as np
import torch
from owlcaribou import infer, metrics, sweep as S

run = session.paths.run_dir(RUN_NAME)
failures = []
for model in MODELS:
    out = run / "sweep" / model
    if not FORCE and oio.is_complete(out, code_hash=session.code_hash, model=model, floor=FLOOR):
        print(f"[cached] {model}")
        continue
    # An old marker must not vouch for a rerun that stops half-way.
    (out / "_SUCCESS.json").unlink(missing_ok=True)

    print(f"=== {model} ===")
    spec = infer.MODEL_SPECS[model]
    net = infer.build_model(model, session.third_party, device=session.device)
    infer.load_checkpoint(net, session.paths.checkpoint(model), device=session.device)

    started = time.perf_counter()
    collected = []
    for name, candidates in infer.run_candidates(net, patches, device=session.device, floor=FLOOR,
                                                 batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
        collected.append((name, candidates))
        if len(collected) == PROBE_PATCHES:
            per_patch = np.mean([len(c) for _, c in collected])
            print(f"   {per_patch:.0f} candidates per patch on the first {PROBE_PATCHES}; "
                  f"about {per_patch * len(patches) / 1e6:.2f} M for the split")
            if per_patch > MAX_CANDIDATES_PER_PATCH:
                raise RuntimeError(f"{per_patch:.0f} candidates per patch is more than the sweep "
                                   f"needs; raise FLOOR (now {FLOOR}) and run again")
    elapsed = time.perf_counter() - started

    table = S.CandidateTable.from_stream(collected, patches.names, spec.prediction_scale)
    table.save(out / "candidates.npz")

    reference = metrics.load_predictions(run / model / "detections.csv")
    check = S.reproduces(table, reference)
    print(f"   {len(table):,} candidates in {elapsed / 60:.1f} min; replay at the evaluation setting: "
          f"{check['identical']} of {check['patches']} patches identical, "
          f"{check['our_detections']} vs {check['reference_detections']} detections")
    session.record(out, model=model, floor=FLOOR, candidates=len(table),
                   seconds=round(elapsed, 1), reproduction=check)
    if check["differing"]:
        # No success marker: a rerun must redo this model rather than skip it.
        print("   differing patches (name, replayed, full run):", check["examples"])
        failures.append((model, check["differing"]))
    else:
        oio.mark_complete(out, code_hash=session.code_hash, model=model, floor=FLOOR,
                          candidates=len(table), reproduced=True)
    del net
    torch.cuda.empty_cache()

if failures:
    raise RuntimeError(f"replay differs from the full run for {failures}. The curves "
                       "would not describe the run that was scored; look at the examples first.")
print("\nAll models replay the full run exactly. Release the GPU and run 06_sweep_curves.")

=== owl-d ===
   59 candidates per patch on the first 32; about 0.15 M for the split
   381,218 candidates in 4.9 min; replay at the evaluation setting: 2607 of 2607 patches identical, 12494 vs 12494 detections
=== caribou-owl-c ===
   789 candidates per patch on the first 32; about 2.06 M for the split
   3,111,123 candidates in 0.2 min; replay at the evaluation setting: 2607 of 2607 patches identical, 13610 vs 13610 detections
=== owl-c ===
   413 candidates per patch on the first 32; about 1.08 M for the split
   2,430,566 candidates in 0.2 min; replay at the evaluation setting: 2607 of 2607 patches identical, 12774 vs 12774 detections
=== owl-t ===


/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


   514 candidates per patch on the first 32; about 1.34 M for the split
   2,738,521 candidates in 0.3 min; replay at the evaluation setting: 2607 of 2607 patches identical, 11976 vs 11976 detections

All models replay the full run exactly. Release the GPU and run 06_sweep_curves.


## What was written

`results/full_cah/sweep/<model>/candidates.npz` holds every local maximum at or
above 1% of its patch's peak: patch, heatmap row and column, float32 score, and
each patch's peak. `provenance.json` records the reproduction check.

The candidate count printed after the first 32 patches is only a guard against
a runaway number of candidates; the first patches in name order are sparse, so
it under-estimates the total, as the next line of each block shows.

**Next:** `06_sweep_curves.ipynb`, which needs no GPU.
